In [11]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd().parent
SRC_DIR = PROJECT_DIR / "src"
RESULTS_DIR = SRC_DIR / "blocking_results"

sys.path.insert(0, str(SRC_DIR))

from build_training_dataset import build_dataset

In [12]:
S1_PATH = SRC_DIR / "sampled_data" / "sample_source1.tsv"
S2_PATH = SRC_DIR / "sampled_data" / "sample_source2.tsv"
S3_PATH = SRC_DIR / "sampled_data" / "sample_source3.tsv"

GROUND_TRUTH_PATH = SRC_DIR / "sampled_data" / "sample_ground_truth.tsv"
CANDIDATE_PAIRS_PATH = RESULTS_DIR / "experiment2_candidate_pairs.tsv"

OUTPUT_DIR = RESULTS_DIR

In [13]:
s1 = pd.read_csv(S1_PATH, sep="\t", dtype=str)
s2 = pd.read_csv(S2_PATH, sep="\t", dtype=str)
s3 = pd.read_csv(S3_PATH, sep="\t", dtype=str)

candidate_pairs = pd.read_csv(
    CANDIDATE_PAIRS_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

candidate_pairs = candidate_pairs[
    ["s1_entity_id", "candidate_entity_id"]
].drop_duplicates()

candidate_pairs["s1_entity_id"] = (
    candidate_pairs["s1_entity_id"].str.strip()
)

candidate_pairs["candidate_entity_id"] = (
    candidate_pairs["candidate_entity_id"].str.strip()
)

print("Candidate pairs:", len(candidate_pairs))
display(candidate_pairs.head())

ground_truth_raw = pd.read_csv(
    GROUND_TRUTH_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

ground_truth = (
    ground_truth_raw
    .rename(columns={
        "source1_entity_id": "s1_entity_id",
    })
    .assign(
        true_entity_id=lambda dataframe: (
            dataframe["matched_entity_ids"]
            .str.split(",")
        )
    )
    .explode("true_entity_id")
)

ground_truth["s1_entity_id"] = (
    ground_truth["s1_entity_id"].str.strip()
)

ground_truth["true_entity_id"] = (
    ground_truth["true_entity_id"].fillna("").str.strip()
)

# Remove S1 records with no matched source entity.
ground_truth = ground_truth[
    ground_truth["true_entity_id"] != ""
][
    ["s1_entity_id", "true_entity_id"]
].drop_duplicates()

print("Ground-truth pairs:", len(ground_truth))
display(ground_truth.head())

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Candidate pairs:", candidate_pairs.shape)
print("Ground truth:", ground_truth.shape)

Candidate pairs: 87865


,s1_entity_id,candidate_entity_id
0,S1-53356671,S3-468658814
1,S1-53356671,S2-904412198
2,S1-53356671,S2-255338248
3,S1-53356671,S3-401976049
4,S1-53356671,S2-454162657


Ground-truth pairs: 3451


,s1_entity_id,true_entity_id
1,S1-219765772,S2-73475891
1,S1-219765772,S2-359968183
1,S1-219765772,S3-363685397
1,S1-219765772,S3-985947781
2,S1-404079969,S2-672526973


S1: (1000, 4)
S2: (9864, 4)
S3: (10842, 4)
Candidate pairs: (87865, 2)
Ground truth: (3451, 2)


In [14]:
required_record_columns = {
    "entity_id",
    "business_name",
    "business_address",
}

for name, dataframe in {
    "s1": s1,
    "s2": s2,
    "s3": s3,
}.items():
    missing = required_record_columns - set(dataframe.columns)

    if missing:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing)}"
        )

required_candidate_columns = {
    "s1_entity_id",
    "candidate_entity_id",
}

missing = required_candidate_columns - set(candidate_pairs.columns)

if missing:
    raise ValueError(
        "Candidate file is missing columns: "
        f"{sorted(missing)}"
    )

required_truth_columns = {
    "s1_entity_id",
    "true_entity_id",
}

missing = required_truth_columns - set(ground_truth.columns)

if missing:
    raise ValueError(
        "Ground-truth file is missing columns: "
        f"{sorted(missing)}"
    )

for dataframe in [s1, s2, s3]:
    dataframe["entity_id"] = dataframe["entity_id"].astype(str)

candidate_pairs = candidate_pairs[
    ["s1_entity_id", "candidate_entity_id"]
].drop_duplicates()

ground_truth = ground_truth[
    ["s1_entity_id", "true_entity_id"]
].drop_duplicates()

In [15]:
print(
    "Unique S1 records with candidates:",
    candidate_pairs["s1_entity_id"].nunique(),
)

print(
    "Unique candidate pairs:",
    len(candidate_pairs),
)

print(
    "Known true pairs:",
    len(ground_truth),
)

Unique S1 records with candidates: 1000
Unique candidate pairs: 87865
Known true pairs: 3451


In [16]:
dataset, train_data, test_data = build_dataset(
    s1=s1,
    s2=s2,
    s3=s3,
    candidate_pairs=candidate_pairs,
    ground_truth=ground_truth,
    output_dir=OUTPUT_DIR,
)

{
  "total_rows": 13804,
  "positive_rows": 3451,
  "negative_rows": 10353,
  "train_rows": 11006,
  "test_rows": 2798,
  "train_positive_rows": 2772,
  "test_positive_rows": 679,
  "test_size": 0.2,
  "negative_to_positive_ratio": 3,
  "split_strategy": "GroupShuffleSplit by s1_entity_id",
  "feature_columns": [
    "name_exact",
    "address_exact",
    "full_exact",
    "name_token_jaccard",
    "address_token_jaccard",
    "full_token_jaccard",
    "name_levenshtein",
    "address_levenshtein",
    "full_levenshtein",
    "name_token_set_ratio",
    "full_token_set_ratio",
    "name_cosine",
    "full_cosine",
    "name_left_length",
    "name_right_length",
    "address_left_length",
    "address_right_length",
    "name_left_tokens",
    "name_right_tokens",
    "address_left_tokens",
    "address_right_tokens",
    "same_country"
  ]
}


In [17]:
print("Full dataset:", dataset.shape)
print("Training data:", train_data.shape)
print("Test data:", test_data.shape)

display(
    dataset["is_same"]
    .value_counts()
    .rename(index={0: "different", 1: "same"})
    .to_frame("rows")
)

Full dataset: (13804, 25)
Training data: (11006, 25)
Test data: (2798, 25)


,rows
is_same,
different,10353
same,3451


In [18]:
train_entities = set(train_data["s1_entity_id"])
test_entities = set(test_data["s1_entity_id"])

assert not train_entities.intersection(test_entities)

print("No S1 entity leakage detected.")

No S1 entity leakage detected.
